# Phase 1 Group C (Test): LLM SFT 快速验证

>
> **任务**: DimASR (给定文本 + Aspect，预测 VA 分数)
> **数据集**: zho_restaurant (繁体中文餐厅评论)

In [ ]:
# 安装 LLM 相关依赖
!pip install peft transformers bitsandbytes accelerate -q

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working")
DATA_ROOT = Path("/kaggle/input/datasets/kintsugi0v0/dimabsa")

TRAIN_FILE = DATA_ROOT / "zho_restaurant_train_alltasks.jsonl"
DEV_FILE = DATA_ROOT / "zho_restaurant_dev_task1.jsonl"

print(f"Train: {TRAIN_FILE.exists()}, Dev: {DEV_FILE.exists()}")

# 一、工具函数

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def explode_quadruplet(data):
    exploded = []
    for item in data:
        for q in item.get('Quadruplet', []):
            try:
                v, a = map(float, q['VA'].split('#'))
                exploded.append({
                    'Text': item['Text'],
                    'Aspect': q.get('Aspect', ''),
                    'Valence': v,
                    'Arousal': a
                })
            except:
                continue
    return exploded

def explode_aspect_va(data):
    exploded = []
    for item in data:
        for av in item.get('Aspect_VA', []):
            try:
                v, a = map(float, av['VA'].split('#'))
                exploded.append({
                    'Text': item['Text'],
                    'Aspect': av.get('Aspect', ''),
                    'Valence': v,
                    'Arousal': a
                })
            except:
                continue
    return exploded

def subsample(data, n, seed=42):
    """随机子采样，加速训练"""
    if len(data) <= n:
        return data
    rng = random.Random(seed)
    return rng.sample(data, n)

print("工具函数定义完成")

# 二、快速验证配置

In [ ]:
# 加速配置
CONFIG = {
    'seed': 42,
    'max_length': 80,              # 回归模式的 prompt 长度（中文短文本足够）
    'sft_max_length': 160,         # SFT 模式：chat template + 评论 + 回答，需要更长
    'batch_size': 8,               # 0.6B 4bit 模型 + 短序列，T4 可跑 8（原版 2）
    'gradient_accumulation_steps': 1,  # 加大 batch 后无需累积（原版 8）
    'learning_rate': 1e-4,
    'epochs': 2,                   # 快速验证 2 轮（原版 3）
    'weight_decay': 0.01,
    'dropout': 0.1,
    'train_samples': 1200,         # 子采样训练集大小（原版全量 8523）
    'eval_samples': 200,           # 子采样验证集大小（原版 685）
}

# 快速验证实验（只用小模型 0.6B）
MODEL_CONFIGS = {
    'Test-C1': {'model': 'Qwen/Qwen3-0.6B', 'type': 'sft_lora'},
    'Test-C2': {'model': 'Qwen/Qwen3-0.6B', 'type': 'regression_lora'},
}

# 预估速度
steps_per_epoch = CONFIG['train_samples'] // CONFIG['batch_size']
total_steps = steps_per_epoch * CONFIG['epochs']
print(f"配置完成!")
print(f"每 epoch 步数: {steps_per_epoch} (原版 4262)")
print(f"总训练步数: {total_steps} (原版 12786)")
print(f"可用实验: {list(MODEL_CONFIGS.keys())}")

# 三、数据格式转换与数据集

In [ ]:
def convert_to_sft_format(data, tokenizer):
    """转换为 SFT 格式"""
    sft_data = []
    for item in data:
        instruction = f"给定文本「{item['Text']}」和方面词「{item['Aspect']}」，预测VA分数。"
        response = f"Valence={item['Valence']:.2f}, Arousal={item['Arousal']:.2f}"
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response}
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        sft_data.append({
            'input': input_text,
            'instruction': instruction,   # 评估时构造纯 prompt 用
            'output': response,
            'text': item['Text'],
            'aspect': item['Aspect'],
            'valence': item['Valence'],
            'arousal': item['Arousal']
        })
    return sft_data

def convert_to_regression_format(data):
    """转换为回归格式"""
    converted = []
    for item in data:
        prompt = f"文本: {item['Text']}\n方面词: {item['Aspect']}\n预测VA分数:"
        converted.append({
            'prompt': prompt,
            'text': item['Text'],
            'aspect': item['Aspect'],
            'valence': item['Valence'],
            'arousal': item['Arousal']
        })
    return converted

print("格式转换函数定义完成")

In [ ]:
class DimASRSFTDataset(Dataset):
    """SFT 数据集：训练用完整对话（padding 位 label=-100）；
    评估额外提供 ① 左 padding 的纯 prompt（批量生成必需）② [valence, arousal] 数值标签"""

    def __init__(self, data, tokenizer, max_length=160):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        # 预计算纯 prompt 的 token ids（user + generation header，不含答案）
        self.prompt_ids = []
        for item in data:
            msgs = [{'role': 'user', 'content': item['instruction']}]
            prompt_text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
            ids = tokenizer(prompt_text, add_special_tokens=False,
                            truncation=True, max_length=max_length)['input_ids']
            self.prompt_ids.append(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(
            item['input'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100   # padding 位置不计 loss

        # 左 padding 的 prompt（批量 generate 必须左 pad）
        p_ids = self.prompt_ids[idx]
        pad = self.max_length - len(p_ids)
        prompt_input_ids = torch.cat([
            torch.full((pad,), self.tokenizer.pad_token_id, dtype=torch.long),
            torch.tensor(p_ids, dtype=torch.long)])
        prompt_attention_mask = torch.cat([
            torch.zeros(pad, dtype=torch.long),
            torch.ones(len(p_ids), dtype=torch.long)])

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'prompt_input_ids': prompt_input_ids,
            'prompt_attention_mask': prompt_attention_mask,
            'va': torch.tensor([item['valence'], item['arousal']], dtype=torch.float),
        }


class DimASRRegressionDataset(Dataset):
    """回归数据集（变长 token，由 collate_fn 动态 padding）"""

    def __init__(self, data, tokenizer, max_length=80):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        print("预 tokenize 中...")
        self.encodings = []
        for item in data:
            enc = self.tokenizer(
                item['prompt'],
                truncation=True,
                max_length=self.max_length,
                padding=False,
                return_tensors=None
            )
            self.encodings.append(enc)
        print(f"预 tokenize 完成: {len(self.encodings)} 条")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.encodings[idx]
        return {
            'input_ids': torch.tensor(enc['input_ids']),
            'attention_mask': torch.tensor(enc['attention_mask']),
            'labels': torch.tensor([item['valence'], item['arousal']], dtype=torch.float)
        }


def collate_fn(batch):
    """动态 padding：input_ids/attention_mask pad 到批内最长；其余张量键直接 stack"""
    max_len = max(x['input_ids'].shape[0] for x in batch)
    pad_id = 0
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        L = x['input_ids'].shape[0]
        pad_len = max_len - L
        input_ids.append(torch.cat([x['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(x['labels'])
    out = {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attention_mask),
        'labels': torch.stack(labels),
    }
    # SFT 附加键（prompt_input_ids / prompt_attention_mask / va 均为定长张量）
    for k in batch[0].keys():
        if k not in out and isinstance(batch[0][k], torch.Tensor):
            out[k] = torch.stack([x[k] for x in batch])
    return out

print("数据集类定义完成")

# 四、模型定义

In [ ]:
def create_model(model_type, model_name, dropout=0.1):
    """创建 LLM 模型"""
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )
    model = get_peft_model(model, lora_config)
    
    if model_type == 'sft_lora':
        model.print_trainable_parameters()
        return model, tokenizer
    
    elif model_type == 'regression_lora':
        # 冻结非 LoRA 参数
        for name, param in model.named_parameters():
            if 'lora' not in name.lower():
                param.requires_grad = False
        
        # 回归头：device 与 dtype 直接与 base 模型对齐（bf16）
        hidden_size = model.config.hidden_size
        device = next(model.parameters()).device
        model_dtype = next(model.parameters()).dtype
        model.regressor = nn.Linear(hidden_size, 2).to(device=device, dtype=model_dtype)
        nn.init.xavier_uniform_(model.regressor.weight)
        nn.init.zeros_(model.regressor.bias)
        # regressor 需要可训练
        model.regressor.weight.requires_grad_(True)
        model.regressor.bias.requires_grad_(True)
        
        return model, tokenizer
    
    else:
        raise ValueError(f"Unknown model type: {model_type}")

print("模型工厂函数定义完成")

# 五、训练与评估函数

In [ ]:
def train_epoch_sft(model, dataloader, optimizer, device, gradient_accumulation_steps=1):
    """SFT 训练（只在回答 token 上有 loss，padding 位已设 -100）"""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(dataloader, desc="训练")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / gradient_accumulation_steps
        loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
    
    return total_loss / len(dataloader)


def train_epoch_regression(model, dataloader, optimizer, device, criterion):
    """回归训练"""
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="训练"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden = outputs.hidden_states[-1][:, -1, :]
        
        va_pred = model.regressor(hidden)
        va_pred = torch.clamp(va_pred.float(), min=1.0, max=9.0)
        
        # dtype 对齐：统一转 float32 计算 loss
        labels = labels.to(dtype=va_pred.dtype)
        loss = criterion(va_pred, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_model(model, dataloader, device, criterion, model_type='regression_lora'):
    """评估模型：统一比较 [valence, arousal] (B,2) 数值张量
    - regression_lora: 标签在 batch['labels'] (B,2)
    - sft_lora: 用左 padding 的 prompt 生成文本并解析数值；标签在 batch['va'] (B,2)
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(dataloader, desc="评估"):
        if model_type == 'regression_lora':
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden = outputs.hidden_states[-1][:, -1, :]
            va_pred = model.regressor(hidden)
        else:
            # SFT：从左 padding 的纯 prompt 生成，再解析数值
            p_ids = batch['prompt_input_ids'].to(device)
            p_mask = batch['prompt_attention_mask'].to(device)
            labels = batch['va'].to(device)
            
            generated = model.generate(
                input_ids=p_ids,
                attention_mask=p_mask,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=_EVAL_TOKENIZER.pad_token_id,
                eos_token_id=_EVAL_TOKENIZER.eos_token_id
            )
            va_pred = parse_va_from_generated(generated[:, p_ids.shape[1]:], device)
        
        va_pred = torch.clamp(va_pred.float(), min=1.0, max=9.0)
        
        labels = labels.to(dtype=va_pred.dtype)
        loss = criterion(va_pred, labels)
        
        total_loss += loss.item()
        all_preds.append(va_pred.cpu())
        all_labels.append(labels.cpu())
    
    preds = torch.cat(all_preds).numpy()
    labels_np = torch.cat(all_labels).numpy()
    rmse = np.sqrt(np.mean((preds - labels_np) ** 2))
    
    return total_loss / len(dataloader), rmse, preds, labels_np


def parse_va_from_generated(generated_ids, device):
    """从 LLM 生成的 token 中解析 Valence/Arousal 数值"""
    import re
    global _EVAL_TOKENIZER
    texts = _EVAL_TOKENIZER.batch_decode(generated_ids, skip_special_tokens=True)
    preds = []
    for t in texts:
        nums = re.findall(r'(\d+\.?\d*)', t)
        if len(nums) >= 2:
            v, a = float(nums[0]), float(nums[1])
        elif len(nums) == 1:
            v, a = float(nums[0]), float(nums[0])
        else:
            v, a = 5.5, 5.5  # 解析失败用均值
        preds.append([v, a])
    return torch.tensor(preds, dtype=torch.float32, device=device)

print("训练和评估函数定义完成")

# 六、主训练函数

In [ ]:
import time

def main(exp_name, model_name, model_type):
    """运行单个实验（快速版）"""
    start_time = time.time()
    set_seed(CONFIG['seed'])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"{'='*50}")
    print(f"实验: {exp_name} [快速模式]")
    print(f"模型: {model_name} ({model_type})")
    print(f"{'='*50}")
    
    # 加载数据 + 子采样
    print("\n[1/5] 加载数据（子采样）...")
    train_data = load_jsonl(DATA_ROOT / "zho_restaurant_train_alltasks.jsonl")
    dev_data = load_jsonl(DATA_ROOT / "zho_restaurant_dev_task1.jsonl")
    
    train_exploded = subsample(explode_quadruplet(train_data), CONFIG['train_samples'])
    dev_exploded = subsample(explode_aspect_va(dev_data), CONFIG['eval_samples'])
    print(f"训练集: {len(train_exploded)} 样本 (全量 8523)")
    print(f"验证集: {len(dev_exploded)} 样本")
    
    # 创建模型
    print("\n[2/5] 创建模型...")
    global _EVAL_TOKENIZER
    model, tokenizer = create_model(model_type, model_name, CONFIG['dropout'])
    _EVAL_TOKENIZER = tokenizer
    model.to(device)
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"可训练参数量: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    
    # 准备数据
    print("\n[3/5] 准备数据...")
    if model_type == 'sft_lora':
        sft_len = CONFIG.get('sft_max_length', CONFIG['max_length'])
        train_formatted = convert_to_sft_format(train_exploded, tokenizer)
        dev_formatted = convert_to_sft_format(dev_exploded, tokenizer)
        train_dataset = DimASRSFTDataset(train_formatted, tokenizer, sft_len)
        dev_dataset = DimASRSFTDataset(dev_formatted, tokenizer, sft_len)
    else:
        train_formatted = convert_to_regression_format(train_exploded)
        dev_formatted = convert_to_regression_format(dev_exploded)
        train_dataset = DimASRRegressionDataset(train_formatted, tokenizer, CONFIG['max_length'])
        dev_dataset = DimASRRegressionDataset(dev_formatted, tokenizer, CONFIG['max_length'])
    
    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
        collate_fn=collate_fn, num_workers=0, pin_memory=True
    )
    dev_loader = DataLoader(
        dev_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=True
    )
    
    # 优化器
    print("\n[4/5] 配置优化器...")
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    criterion = nn.MSELoss()
    
    # 训练
    print("\n[5/5] 开始训练...")
    best_rmse = float('inf')
    
    for epoch in range(CONFIG['epochs']):
        if model_type == 'regression_lora':
            train_loss = train_epoch_regression(model, train_loader, optimizer, device, criterion)
        else:
            train_loss = train_epoch_sft(
                model, train_loader, optimizer, device,
                CONFIG['gradient_accumulation_steps']
            )
        
        dev_loss, dev_rmse, preds, labels = evaluate_model(
            model, dev_loader, device, criterion, model_type
        )
        
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}: "
              f"训练损失={train_loss:.4f} | 验证RMSE={dev_rmse:.4f}")
        
        if epoch == 0:
            print(f"  [调试] 预测值范围: [{preds.min():.2f}, {preds.max():.2f}]")
            print(f"  [调试] 真实值范围: [{labels.min():.2f}, {labels.max():.2f}]")
        
        if dev_rmse < best_rmse:
            best_rmse = dev_rmse
            print(f"  ✓ 最佳模型 (RMSE: {best_rmse:.4f})")
    
    # 保存（只保存 LoRA adapter，更快更小，几 MB 而非几 GB）
    model.save_pretrained(PROJECT_ROOT / f"{exp_name}_adapter")
    
    elapsed = time.time() - start_time
    results = {
        'exp_name': exp_name,
        'model_name': model_name,
        'model_type': model_type,
        'best_rmse': float(best_rmse),
        'trainable_params': int(trainable_params),
        'total_params': int(total_params),
        'elapsed_seconds': round(elapsed, 1),
        'mode': 'test (subsampled)'
    }
    with open(PROJECT_ROOT / f"{exp_name}_results.json", 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"\n{'='*50}")
    print(f"完成! 最佳 RMSE: {best_rmse:.4f} | 耗时: {elapsed/60:.1f} 分钟")
    print(f"{'='*50}")
    
    return best_rmse

print("主训练函数定义完成")

# 七、运行实验

In [ ]:
# 运行单个实验（默认快速验证回归模式）
EXP_NAME = 'Test-C2'  # 可选: Test-C1 (SFT), Test-C2 (回归)

config = MODEL_CONFIGS[EXP_NAME]
main(EXP_NAME, config['model'], config['type'])

In [ ]:
# 批量运行所有快速实验
for exp_name, config in MODEL_CONFIGS.items():
    print(f"\n{'#'*60}")
    main(exp_name, config['model'], config['type'])

# 八、结果汇总

In [ ]:
results = []
for exp in MODEL_CONFIGS.keys():
    path = PROJECT_ROOT / f"{exp}_results.json"
    if not path.exists():
        continue
    with open(path) as f:
        r = json.load(f)
        results.append({
            '实验': exp,
            '方法': r['model_type'].upper(),
            'RMSE': round(r['best_rmse'], 4),
            '耗时(分钟)': round(r['elapsed_seconds']/60, 1),
            '可训练参数': f"{r['trainable_params']/r['total_params']*100:.2f}%"
        })

df = pd.DataFrame(results)
print("=" * 70)
print("Phase 1 Group C 快速验证结果")
print("=" * 70)
print(df.to_string(index=False))
df.to_csv(PROJECT_ROOT / "phase1_groupC_test_results.csv", index=False, encoding='utf-8-sig')